In [2]:
# imports

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [3]:
import google.generativeai
load_dotenv(override=True)
google_api_key = os.getenv('GOOGLE_API_KEY')
MODEL = "gemini-2.5-flash"


In [4]:
# Initialize the OpenAI-compatible client for Gemini
gemini_via_openai_client = OpenAI(
    api_key=google_api_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [5]:
system_message = "You are a helpful assistant for an Airline called FlightAI. "
system_message += "Give short, courteous answers, no more than 1 sentence. "
system_message += "Always be accurate. If you don't know the answer, say so."

In [6]:
city_weather = {"london": "18C", "paris": "14C", "tokyo": "25C", "berlin": "7C"}

def get_weather(weather_city):
    print(f"Tool get_weather called for {weather_city}")
    city = weather_city.lower()
    return city_weather.get(city, "Unknown")


In [7]:
weather_function = {
    "name": "get_weather",
    "description": "Get the weather of destination city. Call this whenever you need to know the weather of that place, for example when a user asks 'What is the weather of that city'",
    "parameters": {
        "type": "object",
        "properties": {
            "weather_city": {
                "type": "string",
                "description": "The weather that the user wants to know",
            },
        },
        "required": ["weather_city"],
        "additionalProperties": False
    }
}

In [8]:
# And this is included in a list of tools:

tools = [
    {"type": "function", "function": weather_function}
    ]

In [9]:
def handle_tool_call_weather(message):
    tool_call = message.tool_calls[0]
    arguments = json.loads(tool_call.function.arguments)
    weather = arguments.get('weather_city')
    city = get_weather(weather)
    response = {
        "role": "tool",
        "content": json.dumps({"weather_city": city,"weather": weather}),
        "tool_call_id": tool_call.id
    }
    return response, city

In [10]:
def chat(message, history):
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = gemini_via_openai_client.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        response, city = handle_tool_call_weather(message)
        messages.append(message)
        messages.append(response)
        response = gemini_via_openai_client.chat.completions.create(model=MODEL, messages=messages)

    
    return response.choices[0].message.content

In [11]:
gr.ChatInterface(fn=chat, type="messages").launch(share= False)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Tool get_weather called for london
